# Étape 2 — Base SQLite et contrôles des données

Objectif : comprendre le contenu de `posts.db` et vérifier les publications collectées avant le nettoyage. Ce notebook lit uniquement la base ; il ne collecte pas et ne modifie aucune publication.

Prérequis : une base initialisée par le collecteur de l'étape 1.

## 1. Chemins et imports

In [ ]:
from pathlib import Path
import sys

# Fonctionne depuis le notebook, le sous-projet ou la racine du projet.
cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "bluesky-news-mining", cwd / "textmining" / "bluesky-news-mining"]
ROOT = next((p.resolve() for p in candidates if (p / "notebooks" / "00_fonctions_base.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvrir le notebook depuis le projet Bluesky ou son dossier notebooks.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATABASE_PATH = ROOT / "posts.db"
print("Sous-projet :", ROOT.name)
print("Base SQLite :", (Path(ROOT.name) / DATABASE_PATH.relative_to(ROOT)).as_posix())

In [ ]:
from contextlib import closing
import sqlite3
import pandas as pd
from IPython.display import display

import json

# Charger les fonctions communes, sans lire ni modifier la base.
base_path = ROOT / "notebooks" / "00_fonctions_base.ipynb"
base_notebook = json.loads(base_path.read_text(encoding="utf-8"))
for definition_cell in base_notebook["cells"]:
    if definition_cell["cell_type"] == "code" and "database-definitions" in definition_cell["metadata"].get("tags", []):
        exec(compile("".join(definition_cell["source"]), str(base_path), "exec"), globals())

pd.set_option("display.max_colwidth", 120)

## 2. Schéma de la table `posts`

`uri` est la clé primaire : elle empêche de stocker plusieurs lignes pour la même publication. `did` désigne l'auteur. Les textes restent bruts, avec leurs URL. `is_reply` et `has_link` sont de simples indicateurs, pas des filtres.

In [ ]:
if not DATABASE_PATH.exists():
    raise FileNotFoundError("Base absente : exécuter d'abord le notebook 01.")
with closing(sqlite3.connect(DATABASE_PATH.as_uri() + "?mode=ro", uri=True)) as connection:
    schema = pd.read_sql_query("PRAGMA table_info(posts)", connection)
display(schema[["name", "type", "notnull", "pk"]])

## 3. Charger les publications

On charge toute la base (`limit=None`) pour ne pas limiter silencieusement les contrôles aux 1 000 dernières publications. `load_posts` ordonne les lignes par instant décroissant et convertit les dates en UTC.

In [ ]:
posts = load_posts(DATABASE_PATH, limit=None)
display(pd.Series({
    "Publications": len(posts),
    "Auteurs distincts": posts["did"].nunique(),
    "Réponses": int(posts["is_reply"].sum()),
    "Posts avec lien": int(posts["has_link"].sum()),
    "Dates absentes ou invalides": int(posts["created_at"].isna().sum()),
}, name="Valeur").to_frame())
print("Première date (UTC) :", posts["created_at"].min())
print("Dernière date (UTC) :", posts["created_at"].max())

## 4. Langues déclarées et intégrité

La présence de `fr` est obligatoire mais d'autres langues peuvent coexister. Ces contrôles vérifient la cohérence des données stockées, sans certifier que chaque texte est réellement en français.

In [ ]:
display(posts["langs"].value_counts(dropna=False).rename_axis("Langues déclarées").to_frame("Publications"))

is_french = posts["langs"].fillna("").str.split(",").map(lambda langs: "fr" in langs).astype(bool)
assert is_french.all(), "Des publications sans déclaration fr sont présentes."
assert posts["uri"].notna().all(), "Une URI manque."
assert posts["uri"].is_unique, "Les URI ne sont pas uniques."
assert posts["did"].notna().all(), "Un auteur manque."
assert posts["is_reply"].isin([0, 1]).all()
assert posts["has_link"].isin([0, 1]).all()
print("Contrôles des langues, URI, auteurs et indicateurs : OK.")
if posts.empty:
    print("La base est vide : les contrôles ne remplacent pas une collecte effective.")

## 5. Aperçu des textes bruts

Les doublons de texte peuvent exister dans la base : seule l'URI est unique. Le dédoublonnage des textes sera effectué en mémoire à l'étape suivante.

In [ ]:
display(posts[["created_at", "text", "langs", "is_reply", "has_link"]].head(10))
print("Textes bruts répétés après leur première occurrence :", int(posts["text"].duplicated().sum()))